# Intro to LangChain
LangChain is a popular framework that allows you to quickly build applications and pipelines of Large Language Models (LLMs). You can use it to create chatbots, RAGs, agents and much more.

The main idea of the library is that we can create a _chain_ of different components to create more complex applications. These _chains_ (you can think of them as pipelines) can be made up of various components such as:
- **Prompts templates**: Prompts templates are templates to generate different type of prompts. Like chat prompts, question answering prompts, etc.
- **LLMs**: Large Language Models are the core of LangChain. You can use any LLM that is compatible with the library, like OpenAI, Hugging Face, LLama, etc.
- **Tools**: Tools are functions that can be used by the LLM to perform specific tasks. For example, you can use a tool to search the web, or to access a database.
- **Agents**: Agents are components that can use LLMs and tools to perform specific tasks. They can be used to create chatbots, **R**etrieval **A**ugumentation **G**eneration (RAGs), etc.
- **Retrievers**: Retrievers are components that can be used to retrieve information from a database or a knowledge base. They can be used to create RAGs, or to retrieve information from a database.  
- **Memory**: Memory is a component that can be used to store information about the conversation. It can be used to create chatbots that can remember previous conversations, or to create RAGs that can remember previous queries.

## Using LLMs in LangChain

LangChain supports a wide range of providers for LLMs, including OpenAI, Hugging Face, Groq,  LLama and many others.

Let's start our exploration of LangChain by using Grog integration. 

### Groq Integration
Groq is a provider of LLMs that offers high-performance inference capabilities. To use Groq with LangChain, you need to set up your API key in the `.env` file. Follow the steps in the README.md file to set up your environment.

In [1]:
# Importiere die notwendigen Bibliotheken
from dotenv import load_dotenv  # Zum Laden der Umgebungsvariablen aus .env Datei
import warnings  # Für das Handling von Warnmeldungen
from langchain_groq import ChatGroq  # Groq LLM Integration für LangChain
from langchain_core.prompts import PromptTemplate  # Klasse zum Erstellen von Prompt-Vorlagen

#### Load Credentials from .env file

In [2]:
# Lade die Umgebungsvariablen aus der .env Datei
# Diese Funktion sucht nach einer .env Datei im aktuellen Verzeichnis
# und lädt alle darin definierten Variablen (z.B. GROQ_API_KEY)
load_dotenv()

True

#### Defining the LLM (Using Groq)

We can define the LLM using the [`ChatGroq`](https://python.langchain.com/docs/integrations/chat/groq/) class from the `langchain_groq` module. 
This class allows us to specify:
+ the model - below we use `llama-3.1-8b-instant`
+ the temperature - we set it to `0.1` for more deterministic responses
+ the maximum tokens - we set it to `512` to limit the response length

In [3]:
# Definiere das Large Language Model (LLM) mit Groq
llm = ChatGroq(
    model="llama-3.1-8b-instant",  # Modell: LLama 3.1 mit 8 Milliarden Parametern (schnelle Version)
    temperature=0.1,  # Niedrige Temperatur = deterministischere/vorhersehbarere Antworten (0.0-1.0)
    max_tokens=2048,  # Maximale Länge der Antwort in Tokens (begrenzt die Ausgabelänge)
)

#### Build prompt template
A prompt is a set of instructions or input provided by a user to an LLM to guide its response. It helps the model understand the context and generate relevant output. In LangChain, we can create a prompt template using the `PromptTemplate` class.

In [4]:
# Erstelle eine Prompt-Vorlage mit einem Platzhalter für die Frage
# {question} wird später dynamisch durch die tatsächliche Frage ersetzt
template = """Question: {abc}

Answer: """

# Initialisiere das PromptTemplate-Objekt
# input_variables definiert, welche Variablen im Template ersetzt werden können
prompt = PromptTemplate(template=template, input_variables=["abc"])

The __input_variables__ are defined in the template using curly braces '{}'. This allows us to dynamically insert values into the template when we use it.

#### Define Chain
A chain is sequence of components that are executed in order to produce a final output. In LangChain, we can use the pipe symbol `|` to define a chain of components. The output of one component is passed as input to the next component in the chain.

In [5]:
# Erstelle eine Chain (Kette) durch Verkettung von Prompt und LLM
# Der Pipe-Operator | verbindet die Komponenten:
# 1. Zuerst wird der Prompt mit den Eingabewerten gefüllt
# 2. Dann wird das gefüllte Prompt an das LLM weitergegeben
chain = prompt | llm

#### Invoke the Chain

In [6]:
# Definiere eine Frage zum Testen der Chain
q = "Why earth has a moon?"

In [7]:
# Führe die Chain aus (invoke = aufrufen)
# Die Frage wird als Dictionary übergeben, wobei der Key "question" ist
# Das Ergebnis ist ein Objekt, das die Antwort des LLM enthält
answer = chain.invoke(input={"abc": q})

In [8]:
# Gib die Antwort aus
# answer.content enthält den Text der Antwort
# strip() entfernt führende/nachfolgende Leerzeichen
print(answer.content.strip())

The formation of the Moon is a widely accepted scientific theory. It is believed that the Moon was created about 4.5 billion years ago, shortly after the formation of the Earth. The most widely accepted theory is the Giant Impact Hypothesis.

According to this theory, a massive object, sometimes referred to as Theia, collided with the early Earth. Theia is thought to have been a Mars-sized planetary object that was formed in the same region of the solar system as the Earth. The collision was so violent that it caused a large portion of the Earth's mantle and crust to be ejected into space.

Over time, the debris from the collision coalesced and formed the Moon. The Moon is thought to have formed in a disk of debris that surrounded the Earth after the collision. The debris in this disk collided and merged, eventually forming the Moon.

The Giant Impact Hypothesis explains many features of the Moon, including its similar composition to the Earth, its relatively small size, and the fact t

-> 2 errors: ... its relatively small size.... moon is the largest rel. to its planet's size // ...why the Moon has a large iron core... moon has a small iron core; an impact would mostly grasp mantle material and not core materal, indicating that Giant Impact Hypothesis is indeed correct.

If we'd like to ask multiple questions we can by passing a list of dictionary objects, where the dictionaries must contain the input variable set in our prompt template ("question") that is mapped to the question we'd like to ask.

In [15]:
# Erstelle eine Liste von Fragen für Batch-Verarbeitung
# Jede Frage ist ein Dictionary mit dem Key "question"
# So können mehrere Fragen auf einmal verarbeitet werden
qs = [ 
    {"abc": "What is the backpropagation algorithm?"},
    {"abc": "What is the purpose of the activation function in a neural network?"},
    {"abc": "What is the difference between supervised and unsupervised learning?"},
    {"abc": "Explain the concept of overfitting in machine learning."},
]

In [16]:
# Verarbeite alle Fragen als Batch (gleichzeitig)
# batch() ist effizienter als einzelne invoke() Aufrufe
# Gibt eine Liste von Antworten zurück (in derselben Reihenfolge wie die Fragen)
answers = chain.batch(qs)

In [18]:
# Iteriere durch alle Fragen und Antworten parallel
# zip() kombiniert die beiden Listen paarweise
for question, answer in zip(qs, answers):
    # Trennlinie für bessere Lesbarkeit (100 Gleichheitszeichen)
    print("=" * 100)
    # Gib die Frage aus (greife auf den Wert im Dictionary zu)
    print(f"Question: {question['abc']}")
    # Gib die Antwort aus (entferne Leerzeichen am Anfang/Ende)
    print(f"Answer: {answer.content.strip()}")
    # Abschließende Trennlinie
    print("=" * 100)

Question: What is the backpropagation algorithm?
Answer: **Backpropagation Algorithm**

The backpropagation algorithm is a widely used method for training artificial neural networks (ANNs). It is an optimization technique used to minimize the error between the network's predictions and the actual output.

**How Backpropagation Works**

The backpropagation algorithm works as follows:

1. **Forward Pass**: The network processes the input data and produces an output.
2. **Error Calculation**: The difference between the predicted output and the actual output is calculated, resulting in an error value.
3. **Backward Pass**: The error is propagated backwards through the network, adjusting the weights and biases of each layer to minimize the error.
4. **Weight Update**: The weights and biases are updated based on the error and the learning rate.

**Mathematical Formulation**

The backpropagation algorithm can be mathematically formulated as follows:

Let `y` be the actual output, `y_pred` be 